### LLaMA Base Inference on Test Data

This file will process the train/test files provided for GPT-4 and then match them with LLaMA files to ensure that we only get the test data from LLaMA and then report the LLaMA-Base scores for test data

In [1]:
import pandas as pd
import json
import evaluate

/data/mn27889/miniconda3/envs/mental-health-agents/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### Reading the Question and Answer Pairs from Test Dataset Phase 2

In [4]:
ques_list = []
ans_list = []
llama_resp_list = []

with open('phase2_data_freedom_intelligence/dataset2_llama3_qa_pairs.jsonl', 'rb') as file:
    for line in file:
        json_object = json.loads(line)
        ques_list.append(json_object['question'])
        ans_list.append(json_object['ground_truth'])
        llama_resp_list.append(json_object['llama_response'])

In [5]:
dataset = pd.DataFrame({'question': ques_list,
                          'answer': ans_list,
                          'llama_response_base': llama_resp_list})
dataset

,question,answer,llama_response_base
0,An 88-year-old woman with osteoarthritis is ex...,Gastric ulcer,Gastrointestinal bleeding in a patient taking ...
1,In the context of disseminated intravascular c...,Fibrin degradation products,Platelet count
2,"In a 3-year-old boy with severe diarrhea, vomi...","Double-stranded, icosahedral, non-enveloped",Norovirus
3,Based on the chest radiograph and abdominal CT...,Hydatid Cyst,Possible diagnoses to consider based on sympto...
4,What is one potential side effect that is not ...,Anaphylaxis,Aplastic anemia
...,...,...,...
40639,What inflammatory condition is associated with...,Ankylosing spondylitis,Ankylosing Spondylitis
40640,What is the term for an agent that inhibits th...,Bacteriostatic,Inhibitory antimicrobial
40641,Which virus is not typically a causative agent...,Mumps virus,Adenovirus
40642,Which disorder has the potential to develop in...,Paroxysmal nocturnal Hemoglobinuria,Myeloproliferative neoplasms (MPNs) such as Po...


#### Reading the Question and Answer Pairs from Test Dataset Phase 2

In [10]:
ques_list = []
ans_list = []
llama_resp_list = []

with open('phase2_data_freedom_intelligence/test_freedom_intelligence.jsonl', 'rb') as file:
    for line in file:
        json_object = json.loads(line)
        ques_list.append(json_object['question'])
        ans_list.append(json_object['answer'])

In [11]:
test_dataset = pd.DataFrame({'question': ques_list,
                          'answer': ans_list})
test_dataset

,question,answer
0,A 59-year-old man has a 5-month history of bre...,Subpleural cystic enlargement
1,A patient presents with photopsia and floaters...,Rhegmetogenous retinal detachment
2,Based on the clinical presentation and the pat...,Drug-induced pulmonary disease
3,A 51-year-old woman presents with weakness tha...,Type II hypersensitivity reaction
4,What clinical finding is most likely to be pre...,Canon A waves
...,...,...
4060,Which anticoagulant does not require routine c...,Dabigatran etexilate
4061,A child presents with perianal itching that di...,Enterobius vermicularis
4062,What is the appropriate investigation to diagn...,Transcranial ultrasound
4063,A 12-month-old boy presents with a history of ...,NADPH oxidase complex


#### Combining the two datasets

Since both the datasets have duplicate questions/answer, we need to only get the llama responses which are present for a unique question. Then combine it with the test dataset so that we have a llama_response for each question in test dataset since it will be ultimately used for later calculations

Firstly, getting the unique set of questions from the main dataset

In [17]:
unique_dataset = dataset.drop_duplicates(subset=['question'])
unique_dataset

,question,answer,llama_response_base
0,An 88-year-old woman with osteoarthritis is ex...,Gastric ulcer,Gastrointestinal bleeding in a patient taking ...
1,In the context of disseminated intravascular c...,Fibrin degradation products,Platelet count
2,"In a 3-year-old boy with severe diarrhea, vomi...","Double-stranded, icosahedral, non-enveloped",Norovirus
3,Based on the chest radiograph and abdominal CT...,Hydatid Cyst,Possible diagnoses to consider based on sympto...
4,What is one potential side effect that is not ...,Anaphylaxis,Aplastic anemia
...,...,...,...
40639,What inflammatory condition is associated with...,Ankylosing spondylitis,Ankylosing Spondylitis
40640,What is the term for an agent that inhibits th...,Bacteriostatic,Inhibitory antimicrobial
40641,Which virus is not typically a causative agent...,Mumps virus,Adenovirus
40642,Which disorder has the potential to develop in...,Paroxysmal nocturnal Hemoglobinuria,Myeloproliferative neoplasms (MPNs) such as Po...


Combning the unique questions with test_dataset in a left join fashion to ensure that we have one-to-one mapping

In [24]:
test_dataset = test_dataset.join(unique_dataset.set_index('question'), on='question', rsuffix='_base', how='left')
test_dataset

,question,answer,answer_base,llama_response_base
0,A 59-year-old man has a 5-month history of bre...,Subpleural cystic enlargement,Subpleural cystic enlargement,Asbestosis
1,A patient presents with photopsia and floaters...,Rhegmetogenous retinal detachment,Rhegmetogenous retinal detachment,Vitreal detachment or vitreomacular detachment...
2,Based on the clinical presentation and the pat...,Drug-induced pulmonary disease,Drug-induced pulmonary disease,The most likely cause of a lung condition char...
3,A 51-year-old woman presents with weakness tha...,Type II hypersensitivity reaction,Type II hypersensitivity reaction,The pathogenesis of this patient's condition a...
4,What clinical finding is most likely to be pre...,Canon A waves,Canon A waves,Left bundle branch block (LBBB)
...,...,...,...,...
4060,Which anticoagulant does not require routine c...,Dabigatran etexilate,Dabigatran etexilate,Warfarin
4061,A child presents with perianal itching that di...,Enterobius vermicularis,Enterobius vermicularis,Enterobius vermicularis (pinworm) infestation
4062,What is the appropriate investigation to diagn...,Transcranial ultrasound,Transcranial ultrasound,Electroencephalogram (EEG) and blood tests for...
4063,A 12-month-old boy presents with a history of ...,NADPH oxidase complex,NADPH oxidase complex,Cystic fibrosis


### Calculating the BLEU Results for Phase 2

LLaMA Response Groundtruth Fine-Tuned

In [25]:
bleu_eval = evaluate.load("bleu")
bleu_results = bleu_eval.compute(predictions=test_dataset['llama_response_base'].to_list(), references=test_dataset['answer'].to_list())
bleu_results

{'bleu': 0.01070762978264386,
 'precisions': [0.05242087967973711,
  0.014513508303268982,
  0.005602686973390857,
  0.0030839002267573695],
 'brevity_penalty': 1.0,
 'length_ratio': 4.746070058175516,
 'translation_length': 76687,
 'reference_length': 16158}

### Calculating the ROUGE Results for Phase 2

LLaMA Response Fine-Tuned

In [26]:
rouge_eval = evaluate.load("rouge")
rouge_results = rouge_eval.compute(predictions=test_dataset['llama_response_base'].to_list(), references=test_dataset['answer'].to_list())
rouge_results

{'rouge1': np.float64(0.13288543314044732),
 'rouge2': np.float64(0.04731370656333955),
 'rougeL': np.float64(0.1284658408468126),
 'rougeLsum': np.float64(0.12895752460240203)}